# 06 — Final results on the test month (June 2026)

**June was used once.** Every choice was made on validation (May) before these predictions were computed:

- features `full_network_plus` and XGBoost settings (absolute error, learning rate 0.1, early stopping on April, refit on Aug–Apr with 10 % more trees);
- quantile models P10 / P90 with conformal offsets from April;
- for the research comparison at 15 min, the graph transformer (`graph`, refit) and the fixed 50/50 average.

1. Headline results, validation vs test
2. Paired comparisons
3. Breakdowns: severity, normal vs disruption days, category, day by day
4. Prediction intervals
5. Findings and decisions

Common subset (points labelled at all three horizons), MAE in min of Δd, day-clustered 95 % CIs; `diff` negative = better.

In [1]:
# 0. Setup: test predictions of the baselines, XGBoost, quantiles and (if run) the graph transformer
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px

from swissdelay import config
from swissdelay.evaluation.metrics import evaluate, mae_diff_ci
from swissdelay.models.baselines import PREDICTIONS_PATH as BASELINE_PATH
from swissdelay.models.quantile import interval_metrics
from swissdelay.viz import GRAY, SERIES, use_style

use_style()
pd.set_option("display.float_format", "{:,.3f}".format)
P, R = config.PROCESSED, config.REPORTS
KEYS = ["trip_key", "stop_seq", "horizon_min"]
U = ", ".join(KEYS)

con = duckdb.connect()
test = con.sql(f"""
SELECT t.*, b.pred_persistence, b.pred_offset,
       q.lo_cal, q.hi_cal, q.lo AS lo_raw, q.hi AS hi_raw, q.pred_q10, q.pred_q90
FROM read_parquet('{P}/tabular_predictions_test.parquet') t
JOIN read_parquet('{BASELINE_PATH}') b USING ({U})
LEFT JOIN read_parquet('{P}/quantile_predictions_test.parquet') q USING ({U})
""").df()
gt_path = P / "gt_graph_refit_predictions_test_h15.parquet"
HAS_GT = gt_path.exists()
if HAS_GT:
    gt = pd.read_parquet(gt_path)[KEYS + ["pred_gt_graph_refit"]]
    test = test.merge(gt, on=KEYS, how="left")
    test["pred_avg"] = (test["pred_gt_graph_refit"] + test["pred_xgb_full_network_plus"]) / 2
for c in ("in_common_subset", "is_disruption_day"):
    test[c] = test[c].astype(bool)
common = test[test["in_common_subset"]].copy()
print(f"{len(test):,} test rows, {len(common):,} in the common subset "
      f"({common.groupby('horizon_min').size().iloc[0]:,} points per horizon), "
      f"{common.loc[common['is_disruption_day'], 'operating_day'].nunique()} disruption days, graph transformer: {HAS_GT}")

917,255 test rows, 526,260 in the common subset (175,420 points per horizon), 13 disruption days, graph transformer: True


## 1. Headline results

In [2]:
# 1.1 Test month, common subset: every model, 3 horizons
MODELS = ["pred_persistence", "pred_offset", "pred_historical", "pred_xgb_full", "pred_xgb_norefit_full_network_plus",
          "pred_xgb_full_network_plus"]
res = evaluate(common, MODELS, reference="pred_historical")
display(res.pivot(index="model", columns="horizon_min", values="mae").loc[[m.removeprefix("pred_") for m in MODELS]])
res[["horizon_min", "model", "mae", "mae_lo", "mae_hi", "p90_abs_err", "diff_vs_ref", "diff_lo", "diff_hi"]]

horizon_min,15,30,60
model,,,
persistence,1.399,1.581,1.797
offset,1.155,1.379,1.649
historical,0.992,1.228,1.488
xgb_full,0.829,1.016,1.222
xgb_norefit_full_network_plus,0.810,0.993,1.206
xgb_full_network_plus,0.805,0.990,1.200


,horizon_min,model,mae,mae_lo,mae_hi,p90_abs_err,diff_vs_ref,diff_lo,diff_hi
0,15,persistence,1.399,1.372,1.423,2.750,0.407,0.395,0.420
1,15,offset,1.155,1.122,1.186,2.400,0.163,0.160,0.167
2,15,historical,0.992,0.956,1.026,2.158,0.000,0.000,0.000
3,15,xgb_full,0.829,0.796,0.861,1.813,-0.163,-0.167,-0.160
4,15,xgb_norefit_full_network_plus,0.810,0.780,0.839,1.746,-0.182,-0.187,-0.176
5,15,xgb_full_network_plus,0.805,0.774,0.834,1.736,-0.187,-0.192,-0.181
6,30,persistence,1.581,1.537,1.621,3.217,0.354,0.342,0.365
7,30,offset,1.379,1.327,1.425,2.983,0.151,0.146,0.156
8,30,historical,1.228,1.175,1.277,2.767,0.000,0.000,0.000
9,30,xgb_full,1.016,0.967,1.063,2.243,-0.211,-0.219,-0.204


In [3]:
# 1.2 Validation vs test: same models, same definitions
val = con.sql(f"""
SELECT v.horizon_min, v.operating_day, v.delta_min, v.in_common_subset, v.pred_historical,
       n.pred_xgb_full, v.pred_xgb_full_network_plus
FROM read_parquet('{P}/tabular_predictions_valid.parquet') v
JOIN read_parquet('{P}/tabular_predictions_valid_network.parquet') n USING ({U})
""").df()
val = val[val["in_common_subset"].astype(bool)]
cmp_models = ["pred_historical", "pred_xgb_full", "pred_xgb_full_network_plus"]
both = pd.concat([evaluate(val, cmp_models, n_boot=200).assign(month="May (validation)"),
                  evaluate(common, cmp_models, n_boot=200).assign(month="June (test)")])
t = both.pivot_table(index="model", columns=["month", "horizon_min"], values="mae")
gain = both.pivot_table(index=["month", "horizon_min"], columns="model", values="mae")
gain["xgb vs historical %"] = 100 * (gain["xgb_full_network_plus"] / gain["historical"] - 1)
display(t.round(3))
gain.round(3)

month                 June (test)             May (validation)            
horizon_min                    15    30    60               15    30    60
model                                                                     
historical                  0.992 1.228 1.488            0.883 1.073 1.279
xgb_full                    0.829 1.016 1.222            0.720 0.860 1.020
xgb_full_network_plus       0.805 0.990 1.200            0.701 0.843 1.005

model                         historical  xgb_full  xgb_full_network_plus  \
month            horizon_min                                                
June (test)      15                0.992     0.829                  0.805   
                 30                1.228     1.016                  0.990   
                 60                1.488     1.222                  1.200   
May (validation) 15                0.883     0.720                  0.701   
                 30                1.073     0.860                  0.843   
                 60                1.279     1.020                  1.005   

model                         xgb vs historical %  
month            horizon_min                       
June (test)      15                       -18.806  
                 30                       -19.411  
                 60                       -19.336  
May (validation) 15                       -20.589  
                 30                       -21.423  
                 60                       -21.408

In [4]:
# 1.3 Chart: MAE by horizon on May and June
fig = px.bar(both, x="horizon_min", y="mae", color="model", barmode="group", facet_col="month",
             color_discrete_map={"historical": SERIES[1], "xgb_full": SERIES[3], "xgb_full_network_plus": SERIES[0]},
             title="MAE by horizon: validation (May) vs test (June), common subset")
fig.update_xaxes(type="category", title="horizon (min)")
fig.update_layout(yaxis_title="MAE (min)", legend_title=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("month=", "")))
fig.show()

> **Finding 1.1–1.3 (headline, June, common subset, 175,420 points per horizon).**
> - **The champion holds on the test month:** XGBoost `full_network_plus` reaches 0.805 / 0.990 / 1.200 min at 15 / 30 / 60 min, which is **−18.8 / −19.4 / −19.3 %** vs the historical median and −42 / −37 / −33 % vs persistence. Every interval of the difference excludes 0.
> - **June is a harder month than May.** All models are worse: the historical baseline goes from 0.883 to 0.992 at 15 min.
> - **The relative gain is almost the same:** −18.8 to −19.4 % in June vs −20.6 to −21.4 % in May. Choosing the model on May didn't overfit to it.
> - The tail behaves the same way: the 90th-percentile absolute error is 1.74 / 2.15 / 2.60 min, against 2.16 / 2.77 / 3.37 for the historical baseline.

## 2. Paired comparisons

In [5]:
# 2.1 Paired MAE differences on the test month (day-clustered CI)
def paired(df, a, b, label, by=None):
    keys = ["horizon_min"] + ([by] if by else [])
    rows = []
    for k, g in df.groupby(keys, observed=True):
        y, day = g["delta_min"].to_numpy(), g["operating_day"].to_numpy()
        ea, eb = np.abs(y - g[a].to_numpy()), np.abs(y - g[b].to_numpy())
        d, lo, hi = mae_diff_ci(day, ea, eb)
        rows.append(dict(zip(keys, k if isinstance(k, tuple) else (k,)))
                    | {"comparison": label, "n": len(g), "mae_ref": eb.mean(), "diff": d, "lo": lo, "hi": hi,
                       "rel_%": 100 * d / eb.mean(), "significant": hi < 0 or lo > 0})
    return pd.DataFrame(rows)

PAIRS = [
    ("pred_xgb_full_network_plus", "pred_historical", "champion vs historical"),
    ("pred_xgb_full_network_plus", "pred_xgb_full", "network features (full_network_plus vs full)"),
    ("pred_xgb_full_network_plus", "pred_xgb_norefit_full_network_plus", "refit vs no refit"),
]
comp = [paired(common, a, b, label) for a, b, label in PAIRS]
if HAS_GT:
    g15 = common[common["horizon_min"] == 15]
    comp += [paired(g15, "pred_gt_graph_refit", "pred_xgb_full_network_plus", "GT refit vs XGBoost (15 min)"),
             paired(g15, "pred_avg", "pred_xgb_full_network_plus", "average vs XGBoost (15 min)"),
             paired(g15, "pred_avg", "pred_gt_graph_refit", "average vs GT (15 min)")]
comp = pd.concat(comp)
comp[["comparison", "horizon_min", "n", "mae_ref", "diff", "lo", "hi", "rel_%", "significant"]].round(4)

,comparison,horizon_min,n,mae_ref,diff,lo,hi,rel_%,significant
0,champion vs historical,15,175420,0.992,-0.187,-0.192,-0.181,-18.806,True
1,champion vs historical,30,175420,1.228,-0.238,-0.247,-0.230,-19.411,True
2,champion vs historical,60,175420,1.488,-0.288,-0.299,-0.276,-19.336,True
0,network features (full_network_plus vs full),15,175420,0.829,-0.023,-0.026,-0.020,-2.793,True
1,network features (full_network_plus vs full),30,175420,1.016,-0.027,-0.032,-0.022,-2.650,True
2,network features (full_network_plus vs full),60,175420,1.222,-0.022,-0.032,-0.015,-1.819,True
0,refit vs no refit,15,175420,0.810,-0.005,-0.006,-0.004,-0.608,True
1,refit vs no refit,30,175420,0.993,-0.004,-0.005,-0.003,-0.366,True
2,refit vs no refit,60,175420,1.206,-0.006,-0.007,-0.004,-0.464,True
0,GT refit vs XGBoost (15 min),15,175420,0.805,-0.000,-0.002,0.002,-0.018,False


> **Finding 2.1 (paired, June).**
> - **Network features:** −0.023 / −0.027 / −0.022 min (−2.8 / −2.7 / −1.8 %), significant at every horizon. That is larger than on May (−0.019 / −0.017 / −0.014).
> - **Refit:** −0.005 / −0.004 / −0.006, significant but smaller than on May (−0.008).
> - **The graph transformer alone does not beat XGBoost on June:** −0.000 [−0.002, +0.002], not significant. Its May advantage (−0.0035) does not hold.
> - **The 50/50 average does:** −0.008 [−0.009, −0.007] vs XGBoost (−1.0 %), and −0.008 vs the GT, both significant.
> - The ensemble result replicates across months; the single-model GT result does not.

## 3. Breakdowns (test, common subset)

In [6]:
# 3.1 By current delay: MAE of historical vs champion, and the network gain
common["d0_bucket"] = pd.cut(common["d0_min"], [-99, 1, 3, 10, 999], right=False,
                             labels=["on time < 1", "minor 1-3", "moderate 3-10", "major > 10"])
by_d0 = evaluate(common, ["pred_historical", "pred_xgb_full", "pred_xgb_full_network_plus"],
                 group_cols=("horizon_min", "d0_bucket"), reference="pred_historical", n_boot=300)
display(by_d0.pivot_table(index="d0_bucket", columns=["horizon_min", "model"], values="mae", observed=True).round(3))
net = paired(common, "pred_xgb_full_network_plus", "pred_xgb_full", "", by="d0_bucket")
net.pivot(index="d0_bucket", columns="horizon_min", values="rel_%").round(1)

horizon_min           15                                        30           \
model         historical xgb_full xgb_full_network_plus historical xgb_full   
d0_bucket                                                                     
major > 10         2.453    2.008                 1.938      3.855    2.930   
minor 1-3          0.963    0.819                 0.798      1.163    1.002   
moderate 3-10      1.459    1.023                 1.005      2.233    1.415   
on time < 1        0.880    0.766                 0.741      1.002    0.897   

horizon_min                                 60                                 
model         xgb_full_network_plus historical xgb_full xgb_full_network_plus  
d0_bucket                                                                      
major > 10                    2.876      5.544    4.220                 4.081  
minor 1-3                     0.971      1.340    1.174                 1.155  
moderate 3-10                 1.384      2.905    1.707                 1.671  
on time < 1                   0.874      1.198    1.081                 1.062

horizon_min,15,30,60
d0_bucket,,,
major > 10,-3.500,-1.900,-3.300
minor 1-3,-2.600,-3.000,-1.600
moderate 3-10,-1.800,-2.200,-2.100
on time < 1,-3.200,-2.600,-1.700


In [7]:
# 3.2 Normal vs disruption days (June has the most disruption days of all splits)
by_dis = evaluate(common, ["pred_historical", "pred_xgb_full", "pred_xgb_full_network_plus"],
                  group_cols=("horizon_min", "is_disruption_day"), reference="pred_historical", n_boot=500)
display(by_dis.pivot_table(index="is_disruption_day", columns=["horizon_min", "model"], values="mae").round(3))
dis = pd.concat([paired(common, "pred_xgb_full_network_plus", "pred_historical", "champion vs historical", by="is_disruption_day"),
                 paired(common, "pred_xgb_full_network_plus", "pred_xgb_full", "network features", by="is_disruption_day")])
dis[["comparison", "horizon_min", "is_disruption_day", "n", "mae_ref", "diff", "lo", "hi", "rel_%", "significant"]].round(4)

horizon_min               15                                        30  \
model             historical xgb_full xgb_full_network_plus historical   
is_disruption_day                                                        
False                  0.953    0.792                 0.771      1.165   
True                   1.045    0.879                 0.853      1.314   

horizon_min                                              60           \
model             xgb_full xgb_full_network_plus historical xgb_full   
is_disruption_day                                                      
False                0.959                 0.937      1.394    1.128   
True                 1.095                 1.062      1.616    1.351   

horizon_min                              
model             xgb_full_network_plus  
is_disruption_day                        
False                             1.115  
True                              1.316

,comparison,horizon_min,is_disruption_day,n,mae_ref,diff,lo,hi,rel_%,significant
0,champion vs historical,15,False,101134,0.953,-0.182,-0.187,-0.177,-19.107,True
1,champion vs historical,15,True,74286,1.045,-0.193,-0.201,-0.184,-18.433,True
2,champion vs historical,30,False,101134,1.165,-0.228,-0.235,-0.222,-19.584,True
3,champion vs historical,30,True,74286,1.314,-0.252,-0.265,-0.239,-19.202,True
4,champion vs historical,60,False,101134,1.393,-0.278,-0.289,-0.268,-19.954,True
5,champion vs historical,60,True,74286,1.616,-0.301,-0.322,-0.280,-18.610,True
0,network features,15,False,101134,0.792,-0.021,-0.023,-0.018,-2.624,True
1,network features,15,True,74286,0.879,-0.026,-0.032,-0.020,-3.002,True
2,network features,30,False,101134,0.959,-0.022,-0.025,-0.019,-2.313,True
3,network features,30,True,74286,1.095,-0.033,-0.043,-0.023,-3.052,True


In [8]:
# 3.3 By category
by_cat = evaluate(common, ["pred_historical", "pred_xgb_full_network_plus"], group_cols=("horizon_min", "category"),
                  reference="pred_historical", n_boot=300)
by_cat.pivot_table(index="category", columns=["horizon_min", "model"], values="mae").round(3)

horizon_min         15                               30                        \
model       historical xgb_full_network_plus historical xgb_full_network_plus   
category                                                                        
EC               1.502                 1.217      2.258                 1.754   
IC               1.134                 0.962      1.355                 1.137   
IR               0.932                 0.740      1.166                 0.916   
RE               0.912                 0.734      1.105                 0.906   

horizon_min         60                        
model       historical xgb_full_network_plus  
category                                      
EC               2.610                 2.057  
IC               1.609                 1.337  
IR               1.411                 1.110  
RE               1.404                 1.167

In [9]:
# 3.4 Day by day at 15 min: MAE of historical and champion, disruption days marked
d = common[common["horizon_min"] == 15]
daily = d.groupby("operating_day").apply(lambda g: pd.Series({
    "historical": np.mean(np.abs(g["delta_min"] - g["pred_historical"])),
    "xgb_full_network_plus": np.mean(np.abs(g["delta_min"] - g["pred_xgb_full_network_plus"])),
    "disruption": g["is_disruption_day"].iloc[0]}), include_groups=False).reset_index()
fig = px.line(daily.melt(id_vars=["operating_day", "disruption"], var_name="model", value_name="mae"),
              x="operating_day", y="mae", color="model", markers=True,
              color_discrete_map={"historical": SERIES[1], "xgb_full_network_plus": SERIES[0]},
              title="Daily MAE at 15 min, June 2026 (shaded: disruption days)")
for day in daily.loc[daily["disruption"].astype(bool), "operating_day"]:
    fig.add_vrect(x0=pd.Timestamp(day) - pd.Timedelta(hours=12), x1=pd.Timestamp(day) + pd.Timedelta(hours=12),
                  fillcolor=GRAY, opacity=0.15, line_width=0)
fig.update_layout(yaxis_title="MAE (min)", xaxis_title=None, legend_title=None)
fig.show()

> **Finding 3.1–3.4 (breakdowns, June).**
> - **By current delay:** the champion's gain over the historical median is largest for **moderately late trains** (3–10 min): −31 / −38 / −42 % at 15 / 30 / 60 min. For major delays it is −21 / −25 / −26 %, and for on-time trains −16 / −13 / −11 %, the same pattern as on May.
>   - The network features help in every delay bucket: −1.6 to −3.5 %.
> - **Normal vs disruption days (13 of 30 days, 42 % of points):**
>   - every model is worse on disruption days (0.853 vs 0.771 at 15 min);
>   - the champion's relative gain over the historical median is the same (−18.4 to −19.2 % on disruption days, −19.1 to −20.0 % on normal days).
> - **The network features matter more on disruption days**, and most at the longest horizon. Network gain by horizon:
>
>   | Horizon | Normal days | Disruption days |
>   |---|---|---|
>   | 15 min | −2.6 % | **−3.0 %** |
>   | 30 min | −2.3 % | **−3.1 %** |
>   | 60 min | −1.1 % | **−2.6 %** |
>
>   This is the first direct evidence for the hypothesis that the state of the network is most useful when things go wrong. May had only one disruption day.
> - **By category:** at 15 min the gain vs the historical median is −19 % for EC (still the hardest, 1.22 min), −21 % for IR, −20 % for RE and −15 % for IC.

## 4. Prediction intervals (P10–P90)

In [10]:
# 4.1 Coverage on the test month: raw and calibrated, by horizon and current delay
q = pd.read_csv(R / "quantile_test.csv")
q.pivot_table(index=["horizon_min", "d0_bucket"], columns="interval",
              values=["coverage", "below_p10", "above_p90", "median_width"], sort=False).round(3)

coverage            below_p10            above_p90  \
interval                       raw calibrated       raw calibrated       raw   
horizon_min d0_bucket                                                          
15          all              0.744      0.771     0.123      0.103     0.133   
            on time < 1      0.748      0.771     0.119      0.102     0.133   
            minor 1-3        0.744      0.769     0.123      0.106     0.132   
            moderate 3-10    0.722      0.772     0.140      0.103     0.139   
            major > 10       0.734      0.814     0.148      0.082     0.118   
30          all              0.742      0.769     0.122      0.101     0.136   
            on time < 1      0.744      0.767     0.119      0.101     0.137   
            minor 1-3        0.744      0.771     0.122      0.102     0.134   
            moderate 3-10    0.719      0.770     0.139      0.101     0.142   
            major > 10       0.737      0.792     0.138      0.096     0.125   
60          all              0.738      0.769     0.116      0.092     0.146   
            on time < 1      0.739      0.768     0.111      0.089     0.150   
            minor 1-3        0.741      0.772     0.119      0.095     0.140   
            moderate 3-10    0.727      0.768     0.131      0.099     0.142   
            major > 10       0.706      0.766     0.141      0.101     0.154   

                                     median_width             
interval                  calibrated          raw calibrated  
horizon_min d0_bucket                                         
15          all                0.126        1.856      1.950  
            on time < 1        0.127        1.732      1.805  
            minor 1-3          0.126        1.915      2.008  
            moderate 3-10      0.125        2.251      2.461  
            major > 10         0.104        4.919      5.495  
30          all                0.130        2.210      2.317  
            on time < 1        0.131        2.004      2.089  
            minor 1-3          0.128        2.300      2.409  
            moderate 3-10      0.129        3.143      3.417  
            major > 10         0.112        7.565      8.400  
60          all                0.138        2.514      2.654  
            on time < 1        0.143        2.291      2.414  
            minor 1-3          0.133        2.632      2.777  
            moderate 3-10      0.133        3.498      3.751  
            major > 10         0.133       10.532     11.611

In [11]:
# 4.2 Calibrated coverage on normal vs disruption days
rows = []
for (h, dis), g in common.dropna(subset=["lo_cal"]).groupby(["horizon_min", "is_disruption_day"]):
    m = interval_metrics(g["delta_min"].to_numpy(), g["lo_cal"].to_numpy(), g["hi_cal"].to_numpy(),
                         g["pred_q10"].to_numpy(), g["pred_q90"].to_numpy())
    rows.append({"horizon_min": h, "disruption_day": dis, **m})
pd.DataFrame(rows)[["horizon_min", "disruption_day", "n", "coverage", "below_p10", "above_p90", "median_width"]]

,horizon_min,disruption_day,n,coverage,below_p10,above_p90,median_width
0,15,False,101134,0.779,0.098,0.122,1.927
1,15,True,74286,0.759,0.110,0.131,1.984
2,30,False,101134,0.778,0.097,0.125,2.274
3,30,True,74286,0.757,0.107,0.135,2.384
4,60,False,101134,0.780,0.090,0.130,2.603
5,60,True,74286,0.755,0.095,0.150,2.740


> **Finding 4.1–4.2 (prediction intervals, June).**
> - **Calibrated P10–P90 coverage is 77.1 / 76.9 / 76.9 %** (target 80 %; raw 74.4 / 74.2 / 73.8 %). Calibration adds about 3 points and widens the intervals by about 5 %.
> - Coverage is uniform across delay buckets (76.7–77.2 % for the three common ones), so the shortfall is not specific to late trains.
> - **The miss is in the upper tail:**
>   - 12.6–13.8 % of trains end up later than P90, against 9.2–10.3 % earlier than P10;
>   - in a more disrupted month, delays build up beyond what the models saw.
> - **Worse on disruption days:** coverage is 75.5–75.9 %, against 77.8–78.0 % on normal days, and up to 15.0 % of trains end later than P90 at 60 min.
> - The offsets were calibrated on April, a calm month. Conformal guarantees only hold when the calibration month resembles the month being predicted.

## 5. Findings and decisions

### Final results (test month June 2026, used once; common subset, 175,420 points per horizon)

| Model | 15 min | 30 min | 60 min |
|---|---|---|---|
| Persistence | 1.399 | 1.581 | 1.797 |
| Constant offset | 1.155 | 1.379 | 1.649 |
| Historical median | 0.992 | 1.228 | 1.488 |
| XGBoost (train, timetable, context) | 0.829 | 1.016 | 1.222 |
| **XGBoost + network features (deployed)** | **0.805** [0.774, 0.834] | **0.990** [0.943, 1.032] | **1.200** [1.134, 1.264] |
| Graph transformer, refit (research) | ≈ 0.805 (−0.000 vs XGBoost, not significant) | — | — |
| Average GT + XGBoost (research) | −0.008 vs XGBoost (−1.0 %) | — | — |

**Intervals (deployed):** calibrated P10–P90, coverage 77 % (target 80 %).

### Answers to the research questions

| Question | Answer (test month) |
|---|---|
| How hard is persistence to beat? | Easy at every horizon: the constant offset (Δd ≈ −0.8 min) alone beats it, and the champion is 33–42 % better |
| Does the train's trajectory help beyond its current delay? | No (notebook 03, ≤ 0.003 min); the sequence model was skipped |
| Does timetable slack explain recovery? | Partly: the only train-level group that adds beyond the current state (notebook 03) |
| Does the network state help? | **Yes:** −1.8 to −2.8 % at every horizon, larger than on May |
| Does an explicit graph beat hand-made network features? | **Not on its own:** the graph transformer equals XGBoost on June (its May edge doesn't hold), but averaging the two helps on both months (−1.0 to −1.3 %). Most of the graph's value is station-level detail, not topology (notebook 05) |
| How do models behave on disruption days? | Everything is harder, but the relative gain holds (−18 to −19 %). **The network features help more on disruption days**, most at 60 min (−2.6 % vs −1.1 % on normal days). Intervals under-cover more (≈ 76 %) |

### Decisions

| # | Topic | Decision |
|---|---|---|
| 1 | Deployed model | **XGBoost `full_network_plus`** at 15 / 30 / 60 min. The GT + XGBoost average gains −1.0 to −1.3 % on both months, but stays below the 2 % promotion rule and would add graph extraction and PyTorch to the nightly pipeline |
| 2 | Intervals | Calibrated P10–P90 is deployed, reported at 77 % coverage on the test month. No re-tuning on June |
| 3 | Production improvements to evaluate in the Jul–Sep replay (past data only) | (a) **rolling recalibration** of the conformal offsets on the last 14 days; (b) **separate offsets for the two tails**, which targets the upper-tail misses |
| 4 | Test month | Closed: no further model choices may use June |